<a href="https://colab.research.google.com/github/KushagraPan/machine-learning-zoomcamp/blob/main/Homework6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'
df = pd.read_csv(url)

print(f"My dataset has {df.shape[0]} rows and {df.shape[1]} columns.")
display(df.head())


My dataset has 10000 rows and 11 columns.


,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [ ]:
# Missing values can break my models, so I will fill them with 0.
df = df.fillna(0)

# Converting object/string columns to categories to save memory and optimize processing
categorical_cols = ['origin', 'fuel_type', 'drivetrain']
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).astype('category')

print("Checking for any remaining missing values:")
print(df.isnull().sum())
display(df.head())


Checking for any remaining missing values:
model_year             0
origin                 0
fuel_type              0
drivetrain             0
num_doors              0
engine_displacement    0
num_cylinders          0
horsepower             0
vehicle_weight         0
acceleration           0
fuel_efficiency_mpg    0
dtype: int64


,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,0.0,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,0.0,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [ ]:
# Calculate and display summary statistics for df_preprocessed
print("Numerical columns summary statistics:")
display(df.describe())

print("\nCategorical columns summary statistics:")
display(df.describe(include=['category']))

Numerical columns summary statistics:


,model_year,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
count,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,1999.549500,3.50400,2268.807000,6.035200,232.131100,4286.754000,18.389400,29.997700
std,14.207925,0.95209,183.447355,0.542578,75.210419,266.212491,3.280959,2.930245
min,1975.000000,2.00000,1590.000000,4.000000,0.000000,3320.000000,0.000000,19.800000
25%,1987.000000,3.00000,2150.000000,6.000000,234.000000,4110.000000,18.000000,28.000000
50%,2000.000000,4.00000,2270.000000,6.000000,252.000000,4280.000000,18.800000,30.000000
75%,2012.000000,4.00000,2390.000000,6.000000,268.000000,4470.000000,19.700000,31.900000
max,2024.000000,5.00000,3110.000000,8.000000,334.000000,5460.000000,23.800000,41.200000



Categorical columns summary statistics:


,origin,fuel_type,drivetrain
count,10000,10000,10000
unique,3,3,3
top,USA,Gasoline,Front-wheel drive
freq,4202,6913,5519


In [ ]:
# I'll use a 60% / 20% / 20% split ratio with a fixed random state for reproducibility.
from sklearn.model_selection import train_test_split

# First, set aside 20% of the data for the final test set
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)

# Next, split the remaining 80% to get 60% train and 20% validation
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

print(f"My Splits Summary:")
print(f"- Train size: {df_train.shape[0]} rows")
print(f"- Validation size: {df_val.shape[0]} rows")
print(f"- Test size: {df_test.shape[0]} rows")


My Splits Summary:
- Train size: 6000 rows
- Validation size: 2000 rows
- Test size: 2000 rows


In [ ]:
# I will convert my feature dictionaries to a sparse matrix using DictVectorizer.
from sklearn.feature_extraction import DictVectorizer

# Separate features from target variable
features = [col for col in df_train.columns if col != 'fuel_efficiency_mpg']

# Convert DataFrames to lists of dicts
train_dicts = df_train[features].to_dict(orient='records')
val_dicts = df_val[features].to_dict(orient='records')
test_dicts = df_test[features].to_dict(orient='records')

# Fit vectorizer and transform feature sets
dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)
X_test = dv.transform(test_dicts)

# Keep targets separate
y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values

print(f"Encoded features: {len(dv.get_feature_names_out())}")


Encoded features: 16


In [ ]:
# I'll use a single split decision tree to see my initial baseline performance.
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import root_mean_squared_error

dt = DecisionTreeRegressor(max_depth=1, random_state=1)
dt.fit(X_train, y_train)

rmse_train = root_mean_squared_error(y_train, dt.predict(X_train))
rmse_val = root_mean_squared_error(y_val, dt.predict(X_val))

print(f"Baseline Decision Tree RMSE:")
print(f"- Train RMSE: {rmse_train:.4f}")
print(f"- Validation RMSE: {rmse_val:.4f}")


Baseline Decision Tree RMSE:
- Train RMSE: 2.6406
- Validation RMSE: 2.6726


In [ ]:
from sklearn.tree import export_text

# Export the decision rules of our trained model
tree_rules = export_text(dt, feature_names=list(dv.get_feature_names_out()))
print(tree_rules)

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



In [ ]:
encoded_features = dv.get_feature_names_out()
encoded_features

array(['acceleration', 'drivetrain=All-wheel drive',
       'drivetrain=Front-wheel drive', 'drivetrain=Rear-wheel drive',
       'engine_displacement', 'fuel_type=Diesel', 'fuel_type=Gasoline',
       'fuel_type=Hybrid', 'horsepower', 'model_year', 'num_cylinders',
       'num_doors', 'origin=Asia', 'origin=Europe', 'origin=USA',
       'vehicle_weight'], dtype=object)

In [ ]:
# I will start with 10 trees to see if an ensemble performs better than a single split tree.
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

rmse_train_rf = root_mean_squared_error(y_train, rf.predict(X_train))
rmse_val_rf = root_mean_squared_error(y_val, rf.predict(X_val))

print(f"Initial Random Forest Results:")
print(f"- Train RMSE: {rmse_train_rf:.4f}")
print(f"- Validation RMSE: {rmse_val_rf:.4f}")


Initial Random Forest Results:
- Train RMSE: 0.7715
- Validation RMSE: 1.8371


In [ ]:
# I want to see if adding more trees helps reduce my validation error.
n_estimators_values = [10, 50, 100, 150]

for n in n_estimators_values:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    print(f"Using n_estimators = {n} -> Validation RMSE: {rmse:.3f}")


Using n_estimators = 10 -> Validation RMSE: 1.837
Using n_estimators = 50 -> Validation RMSE: 1.772
Using n_estimators = 100 -> Validation RMSE: 1.768
Using n_estimators = 150 -> Validation RMSE: 1.769


In [ ]:
# I'll loop over various max_depth values to find the sweet spot and reduce overfitting.
max_depth_values = [10, 15, 20, 25]
n_estimators_values = [10, 50, 100, 150]

for depth in max_depth_values:
    scores = []
    for n in n_estimators_values:
        rf = RandomForestRegressor(
            n_estimators=n,
            max_depth=depth,
            random_state=1,
            n_jobs=-1
        )
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)
        scores.append(rmse)

    # Checking the average performance across different depths
    mean_rmse = np.mean(scores)
    print(f"For max_depth = {depth} -> Average Validation RMSE: {mean_rmse:.3f}")


For max_depth = 10 -> Average Validation RMSE: 1.755
For max_depth = 15 -> Average Validation RMSE: 1.784
For max_depth = 20 -> Average Validation RMSE: 1.788
For max_depth = 25 -> Average Validation RMSE: 1.786


In [ ]:
rf = RandomForestRegressor(
    n_estimators=10,
    max_depth=20,
    random_state=1,
    n_jobs=-1
)

rf.fit(X_train, y_train)

RandomForestRegressor(max_depth=20, n_estimators=10, n_jobs=-1, random_state=1)

In [ ]:
# I want to know which features my random forest model relies on the most.
feature_names = dv.get_feature_names_out()

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': rf.feature_importances_
})

# Sort features to see the most impactful ones at the top
importance_df = importance_df.sort_values(by='importance', ascending=False)
importance_df


,feature,importance
9,model_year,0.294963
15,vehicle_weight,0.203022
6,fuel_type=Gasoline,0.156192
8,horsepower,0.076004
4,engine_displacement,0.063826
0,acceleration,0.056344
14,origin=USA,0.041750
7,fuel_type=Hybrid,0.020320
2,drivetrain=Front-wheel drive,0.020181
11,num_doors,0.019419


In [ ]:
features_to_check = [
    'vehicle_weight',
    'horsepower',
    'acceleration',
    'engine_displacement'
]

importance_df[
    importance_df['feature'].isin(features_to_check)
]

,feature,importance
15,vehicle_weight,0.203879
8,horsepower,0.078017
4,engine_displacement,0.061005
0,acceleration,0.055201


In [ ]:
# Training an XGBoost Regressor
# I'll build an XGBoost model. First, I need to clean the feature names to meet DMatrix specifications.
import xgboost as xgb

features = list(dv.get_feature_names_out())
clean_features = [f.replace('=', '_').replace(' ', '_').replace('-', '_') for f in features]

# Creating the specialized DMatrix datasets for training and validation
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=clean_features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=clean_features)

# Set up a watchlist to monitor real-time training and validation error
watchlist = [(dtrain, 'train'), (dval, 'val')]

# Initial baseline parameters for my gradient booster
xgb_params = {
    'eta': 0.3,
    'max_depth': 6,
    'min_child_weight': 1,
    'objective': 'reg:squarederror',
    'nthread': 8,
    'seed': 1,
    'verbosity': 1,
}

# Fit the model over 100 rounds
print("Training XGBoost...")
model = xgb.train(
    xgb_params,
    dtrain,
    num_boost_round=100,
    evals=watchlist,
    verbose_eval=10
)


Training XGBoost...
[0]	train-rmse:2.40653	val-rmse:2.46782
[10]	train-rmse:1.47125	val-rmse:1.73205
[20]	train-rmse:1.39368	val-rmse:1.73766
[30]	train-rmse:1.33243	val-rmse:1.74548
[40]	train-rmse:1.25326	val-rmse:1.76296
[50]	train-rmse:1.18645	val-rmse:1.77517
[60]	train-rmse:1.10265	val-rmse:1.79401
[70]	train-rmse:1.04604	val-rmse:1.80444
[80]	train-rmse:0.99487	val-rmse:1.81308
[90]	train-rmse:0.94986	val-rmse:1.82135
[99]	train-rmse:0.90106	val-rmse:1.82646


In [ ]:
# Evaluate the final predictions on the validation set
y_pred_xgb = model.predict(dval)
rmse_xgb = root_mean_squared_error(y_val, y_pred_xgb)
print(f"Final XGBoost (eta=0.3) Validation RMSE: {rmse_xgb:.4f}")


Final XGBoost (eta=0.3) Validation RMSE: 1.8265


In [ ]:
# Tuning the Learning Rate (eta) in XGBoost
# I will experiment with different learning rates to see which one performs the best on validation data.
eta_values = [0.3, 0.1, 0.05, 0.01]

for eta in eta_values:
    print(f"\nEvaluating XGBoost with eta = {eta}...")

    params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,
        'objective': 'reg:squarederror',
        'nthread': 8,
        'seed': 1,
        'verbosity': 0
    }

    # Train the model over 100 rounds
    xgb_model = xgb.train(
        params,
        dtrain,
        num_boost_round=100,
        evals=[(dtrain, 'train'), (dval, 'val')],
        verbose_eval=False
    )

    # Predict and record the performance
    preds = xgb_model.predict(dval)
    rmse = root_mean_squared_error(y_val, preds)
    print(f"Learning Rate (eta): {eta:.2f} -> Validation RMSE: {rmse:.4f}")



Evaluating XGBoost with eta = 0.3...
Learning Rate (eta): 0.30 -> Validation RMSE: 1.8265

Evaluating XGBoost with eta = 0.1...
Learning Rate (eta): 0.10 -> Validation RMSE: 1.7248

Evaluating XGBoost with eta = 0.05...
Learning Rate (eta): 0.05 -> Validation RMSE: 1.7154

Evaluating XGBoost with eta = 0.01...
Learning Rate (eta): 0.01 -> Validation RMSE: 1.9853


### Evaluating Our Best Model on the Test Set
Now that I've completed hyperparameter tuning, our champion model is the XGBoost Regressor with a learning rate (`eta`) of `0.05` (validation RMSE = `1.7154`). To conclude my personal study log, I will evaluate this optimal configuration on our unseen test dataset to see how well our pipeline generalizes to completely new, real-world data.

In [ ]:
# Re-training our best XGBoost model configuration
best_params = {
    'eta': 0.05,
    'max_depth': 6,
    'min_child_weight': 1,
    'objective': 'reg:squarederror',
    'nthread': 8,
    'seed': 1,
    'verbosity': 0
}

# Train our final model on the training set
final_model = xgb.train(
    best_params,
    dtrain,
    num_boost_round=100
)

# Prepare our unseen test data matrix
dtest = xgb.DMatrix(X_test, label=y_test, feature_names=clean_features)

# Predict on the test set
y_pred_test = final_model.predict(dtest)

# Calculate and display the generalization performance
test_rmse = root_mean_squared_error(y_test, y_pred_test)
print(f"Final Best Model (XGBoost, eta=0.05) Test RMSE: {test_rmse:.4f}")

Final Best Model (XGBoost, eta=0.05) Test RMSE: 1.6426
